In [3]:
with open("data.txt", "w") as f:
    f.write("Artificial intelligence is transforming the world.\n")
    f.write("Machine learning is a subset of AI.\n")
    f.write("Fine-tuning LLMs is fun and useful.\n")
    f.write("Neural networks learn patterns from data.\n")
    f.write("Natural language processing helps computers understand text.\n")

In [4]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)

# 1. Load custom text dataset (data.txt)
dataset = load_dataset(
    "text",
    data_files={"train": "data.txt"}
)

# 2. Load GPT-2 tokenizer and model
model_name = "openai-community/gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
model.config.pad_token_id = tokenizer.pad_token_id

# 3. Tokenize dataset
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)

# 4. Prepare language modeling data
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# 5. Training configuration
training_args = TrainingArguments(
    output_dir="./gpt2-custom",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

# 6. Fine-tune GPT-2
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator,
)

trainer.train()

# 7. Save model and tokenizer
trainer.save_model("./gpt2-custom")
tokenizer.save_pretrained("./gpt2-custom")

# 8. Generate text
model.eval()
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
model.to(device)

prompt = "Artificial intelligence is"

inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to(device)

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        top_p=0.95,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

print("\nGenerated Text:")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Generating train split: 0 examples [00:00, ? examples/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Generated Text:
Artificial intelligence is a relatively new field and there are still many unknowns.

